In [1]:
# [셀 1] 데이터 로드 + 피처/타깃 정의 + categorical 선언
import pandas as pd
import numpy as np
import lightgbm as lgb   # pip install lightgbm scikit-learn

OUT_DIR = r'C:\Users\kjlee\도심관련\지하철관련'

train = pd.read_csv(f'{OUT_DIR}\\train.csv', index_col='ts', parse_dates=['ts'])
val   = pd.read_csv(f'{OUT_DIR}\\val.csv',   index_col='ts', parse_dates=['ts'])
test  = pd.read_csv(f'{OUT_DIR}\\test.csv',  index_col='ts', parse_dates=['ts'])

FEATURES = ['hour', 'slot', 'dow', 'month', 'is_weekend', 'is_holiday',
            'is_major_holiday', 'is_minor_holiday', 'holiday_prev', 'holiday_next',
            'is_sandwich', 'is_yearend']
TARGETS  = [c for c in train.columns if c.startswith('y_')]

X_tr, X_va, X_te = train[FEATURES].copy(), val[FEATURES].copy(), test[FEATURES].copy()

# dow/month는 '순서 있는 숫자'가 아니라 '범주' -> categorical 선언
# 트리가 {월,금} vs {화수목}, {12월,1월} vs 나머지 같은 임의 묶음 분기를 할 수 있게 됨
# (실험 결과: 정수 65.4 -> categorical 60.7, 원핫보다도 나음)
for c in ['dow', 'month']:
    X_tr[c] = X_tr[c].astype('category')
    X_va[c] = X_va[c].astype('category')
    X_te[c] = X_te[c].astype('category')

print(f'train {len(train):,} / val {len(val):,} / test {len(test):,}')
print(f'피처 {len(FEATURES)}개, 타깃 {len(TARGETS)}개')

train 126,357 / val 21,252 / test 21,252
피처 12개, 타깃 8개


In [2]:
import joblib, os
MODEL_DIR = f'{OUT_DIR}\\models'
os.makedirs(MODEL_DIR, exist_ok=True)   # 폴더 없으면 생성, 있으면 그냥 통과

MODEL_PATH = f'{MODEL_DIR}\\models_v3.pkl'
models = joblib.load(MODEL_PATH)

In [3]:
# [평가 셀 1] 기본 성적표 — test 전체에서 예측 vs 정답
# 각 타깃: 모델로 test 21,252개 시점을 전부 예측 → 정답과의 차이를 집계

table1 = []
for target in TARGETS:
    pred = models[target].predict(X_te)        # test 전체 예측
    err  = np.abs(pred - test[target])         # 시점별 |예측 - 정답|

    table1.append({
        'target': target,
        'MAE': round(err.mean(), 1),                              # 평균 몇 명 틀리나
        '실제평균': round(test[target].mean(), 1),                 # 문제의 규모
        '상대오차(%)': round(err.mean() / test[target].mean() * 100, 1),  # 규모 대비 %
    })

table1 = pd.DataFrame(table1)
table1

,target,MAE,실제평균,상대오차(%)
0,y_board_5,10.9,50.4,21.7
1,y_board_15,22.4,151.1,14.8
2,y_board_30,35.5,302.2,11.7
3,y_board_60,61.2,604.3,10.1
4,y_alight_5,9.4,48.3,19.5
5,y_alight_15,18.9,144.9,13.0
6,y_alight_30,31.8,289.8,11.0
7,y_alight_60,57.5,578.8,9.9


In [4]:
# [평가 셀 2] 날 성격별 성적표 — 평상일 vs 특이일
#
# 특이일 = 공휴일이거나, 공휴일 바로 앞뒤 날이거나, 샌드위치데이거나, 연말(12/24~31)
# 확장 피처가 노린 날들이 얼마나 잡혔는지, 남은 약점이 어디인지 보는 표

special = ((test.is_holiday == 1) | (test.holiday_prev == 1) | (test.holiday_next == 1)
           | (test.is_sandwich == 1) | (test.is_yearend == 1))

print(f'특이일 비중: {special.mean():.0%} ({special.sum():,}행 / {len(test):,}행)')

table2 = []
for target in TARGETS:
    pred = models[target].predict(X_te)
    err  = np.abs(pred - test[target])

    table2.append({
        'target': target,
        'MAE_전체':    round(err.mean(), 1),
        'MAE_평상일':  round(err[~special].mean(), 1),   # ~ = 특이일이 아닌 행만
        'MAE_특이일':  round(err[special].mean(), 1),
        '상대_평상일(%)': round(err[~special].mean() / test[target][~special].mean() * 100, 1),
        '상대_특이일(%)': round(err[special].mean() / test[target][special].mean() * 100, 1),
    })

table2 = pd.DataFrame(table2)
table2

특이일 비중: 18% (3,927행 / 21,252행)


,target,MAE_전체,MAE_평상일,MAE_특이일,상대_평상일(%),상대_특이일(%)
0,y_board_5,10.9,10.8,11.7,20.8,26.1
1,y_board_15,22.4,21.3,27.2,13.7,20.2
2,y_board_30,35.5,33.3,45.4,10.7,16.9
3,y_board_60,61.2,55.8,84.9,9.0,15.8
4,y_alight_5,9.4,8.9,11.8,18.0,27.2
5,y_alight_15,18.9,17.6,24.5,11.9,18.8
6,y_alight_30,31.8,29.3,42.9,9.9,16.5
7,y_alight_60,57.5,52.3,80.6,8.8,15.5


In [5]:
# [평가 셀 3] 시간대별 성적표 — 피크 vs 비피크
# 피크 = 출근(07~09시) + 퇴근(17~19시). 혼잡 관리 관점에서 가장 중요한 시간대

peak = test.hour.isin([7, 8, 9, 17, 18, 19])
print(f'피크 비중: {peak.mean():.0%} ({peak.sum():,}행)')

table3 = []
for target in TARGETS:
    pred = models[target].predict(X_te)
    err  = np.abs(pred - test[target])
    table3.append({
        'target': target,
        'MAE_피크':   round(err[peak].mean(), 1),
        'MAE_비피크': round(err[~peak].mean(), 1),
        '상대_피크(%)':   round(err[peak].mean() / test[target][peak].mean() * 100, 1),
        '상대_비피크(%)': round(err[~peak].mean() / test[target][~peak].mean() * 100, 1),
    })

table3 = pd.DataFrame(table3)
table3

피크 비중: 31% (6,624행)


,target,MAE_피크,MAE_비피크,상대_피크(%),상대_비피크(%)
0,y_board_5,12.5,10.2,21.4,21.9
1,y_board_15,24.4,21.5,13.9,15.3
2,y_board_30,37.1,34.8,10.6,12.4
3,y_board_60,62.4,60.6,8.9,10.8
4,y_alight_5,12.3,8.1,20.2,19.0
5,y_alight_15,23.2,16.9,12.8,13.2
6,y_alight_30,38.5,28.8,10.7,11.2
7,y_alight_60,68.4,52.5,9.7,10.1
